In [14]:
from collections import defaultdict
import os
import json
import numpy as np
import jax.numpy as jnp

from helper import get_triplets, animate_rod

folder_path = "data"

sim, state0, xf0, K, get_aux = get_triplets()
empty_aux = get_aux(jnp.empty(0))


def reformat_xyztheta(x):
    padding_size = (4 - (len(x) % 4)) % 4
    padded_x = np.append(x, [np.nan] * padding_size)
    reshaped = padded_x.reshape(-1, 4)
    xyzs = reshaped[:, :3].flatten()
    thetas = reshaped[:-1, 3]
    return np.concat([xyzs, thetas])


def get_xb_m(xs):
    data = xs
    lambdas = np.linspace(0, 1.0, data.shape[0])
    xbs = data[:, empty_aux.idx_b]
    fits = np.polyfit(lambdas, xbs, 1)
    m, _ = np.where(np.abs(fits) < 1e-10, 0, fits)
    return m


with open("lambda=0.json", "r") as f:
    data = json.load(f)
    x0 = reformat_xyztheta(data["rod_data"]["x"])

trajectories = defaultdict(list)
for filename in os.listdir(folder_path):
    if filename.endswith(".json"):
        with open(os.path.join(folder_path, filename), "r") as f:
            data = json.load(f)
            disp = np.array(data["endDisp"])
            mag = np.linalg.norm(disp)
            unit_vec = tuple(np.round(disp / mag, decimals=6))
            trajectories[unit_vec].append((mag, data))

final_trajectories = []
for unit_vec in trajectories:
    trajectories[unit_vec].sort(key=lambda x: x[0])
    assert len(trajectories[unit_vec]) == 10
    x = np.asarray(
        [x0]
        + [reformat_xyztheta(sim[1]["rod_data"]["x"]) for sim in trajectories[unit_vec]]
    )
    xf_stars = x[:, empty_aux.idx_f]
    xb_m = get_xb_m(x)
    final_trajectories.append({"xf_stars": xf_stars, "xb_m": xb_m})

print(f"Assembled {len(final_trajectories)} trajectories.")

Assembled 53 trajectories.


In [27]:
i = 0
xf_stars = final_trajectories[i]["xf_stars"]
lambdas = jnp.linspace(0, 1.0, xf_stars.shape[0])
aux = get_aux(final_trajectories[i]["xb_m"])
animate_rod(sim, lambdas, xf_stars, aux)